# Module 03 — AI Agents
## Lesson 7 — Stopping Conditions and Budgets

**Core principle:** the model may decide that it is finished; the application decides whether it is allowed to continue.

This notebook uses deterministic examples so we can inspect termination logic without spending API calls.


## Completion is only one stopping condition

Production agents also stop because of host-owned limits: steps, tool calls, elapsed time, tokens, cost, repeated actions, lack of progress, cancellation, permissions, or approval boundaries. Budgets are ceilings, not targets.


## Independent budgets

- **Steps:** model decision turns.
- **Tool calls:** external actions.
- **Wall-clock time:** total run deadline.
- **Tokens / cost:** provider resource usage.
- **Progress:** repeated actions or iterations that add no useful evidence.
- **Authority:** cancellation, permission, or approval boundaries.


In [ ]:
from dataclasses import dataclass, field
from enum import Enum
import time

class StopReason(str, Enum):
    COMPLETED = "completed"
    STEP_BUDGET = "step_budget"
    TOOL_BUDGET = "tool_budget"
    TIME_BUDGET = "time_budget"
    TOKEN_BUDGET = "token_budget"
    COST_BUDGET = "cost_budget"
    REPEATED_ACTION = "repeated_action"
    NO_PROGRESS = "no_progress"

@dataclass(frozen=True)
class RunBudget:
    max_steps: int = 5
    max_tool_calls: int = 6
    max_seconds: float = 30.0
    max_tokens: int = 8_000
    max_cost_usd: float = 0.25
    max_same_action: int = 2
    max_no_progress: int = 2

@dataclass
class RunUsage:
    steps: int = 0
    tool_calls: int = 0
    tokens: int = 0
    cost_usd: float = 0.0
    no_progress: int = 0


## Centralise budget checks

The host evaluates limits before expensive operations. The model does not get to increase these values.


In [ ]:
def check_budget(
    budget: RunBudget,
    usage: RunUsage,
    *,
    started_at: float,
    now: float | None = None,
) -> StopReason | None:
    current = time.monotonic() if now is None else now
    if current - started_at >= budget.max_seconds:
        return StopReason.TIME_BUDGET
    if usage.steps >= budget.max_steps:
        return StopReason.STEP_BUDGET
    if usage.tool_calls >= budget.max_tool_calls:
        return StopReason.TOOL_BUDGET
    if usage.tokens >= budget.max_tokens:
        return StopReason.TOKEN_BUDGET
    if usage.cost_usd >= budget.max_cost_usd:
        return StopReason.COST_BUDGET
    if usage.no_progress >= budget.max_no_progress:
        return StopReason.NO_PROGRESS
    return None


## Deterministic budget checks

Injecting `now` means time-budget tests do not need to sleep.


In [ ]:
budget = RunBudget(max_steps=3, max_tool_calls=2, max_seconds=10)
usage = RunUsage(steps=3)
print(check_budget(budget, usage, started_at=100.0, now=101.0))

usage = RunUsage()
print(check_budget(budget, usage, started_at=100.0, now=111.0))


## Repeated-action detection

A run can remain under its step limit while repeatedly asking for the same useless action. Fingerprint a tool name plus normalized arguments before executing it again.


In [ ]:
import json
from collections import Counter

def action_fingerprint(name: str, arguments: dict) -> str:
    normalized = json.dumps(arguments, sort_keys=True, separators=(",", ":"))
    return f"{name}:{normalized}"

def repeated_too_often(fingerprints: list[str], *, max_same_action: int) -> bool:
    if not fingerprints:
        return False
    counts = Counter(fingerprints)
    return counts[fingerprints[-1]] > max_same_action

history = [
    action_fingerprint("get_weather", {"city": "Melbourne", "date": "2026-09-26"}),
    action_fingerprint("get_weather", {"date": "2026-09-26", "city": "Melbourne"}),
    action_fingerprint("get_weather", {"city": "Melbourne", "date": "2026-09-26"}),
]
print(repeated_too_often(history, max_same_action=2))


## Repetition and lack of progress are related but different

Polling the same deployment twice may be legitimate if the observation changes. Repeated weather calls with unchanged arguments and unchanged results are usually waste. Start with obvious deterministic loop detection, then add domain-specific progress signals when needed.


## Return a structured stop result

Budget exhaustion is expected control flow, not necessarily an exception. Preserve the reason, usage, and observations so callers can return a partial answer, checkpoint the run, or escalate.


In [ ]:
@dataclass
class RunResult:
    stop_reason: StopReason
    answer: str | None = None
    usage: RunUsage = field(default_factory=RunUsage)
    observations: list[dict] = field(default_factory=list)

def stopped_result(reason: StopReason, *, usage: RunUsage, observations: list[dict]) -> RunResult:
    return RunResult(stop_reason=reason, usage=usage, observations=observations)


## Token and cost accounting

Provider usage is observed after calls. Your application compares it with its own budget. For strict enforcement, reserve or estimate capacity before a call and reconcile with actual usage after it. Cost should be derived from usage and current model pricing/configuration, not from model prose.


## Skeleton of a bounded agent loop

Use this policy around the Lesson 3 agent loop. Check before model calls and again before tool execution.


In [ ]:
def run_bounded_agent(goal: str, budget: RunBudget) -> RunResult:
    started_at = time.monotonic()
    usage = RunUsage()
    observations: list[dict] = []
    fingerprints: list[str] = []

    while True:
        reason = check_budget(budget, usage, started_at=started_at)
        if reason:
            return stopped_result(reason, usage=usage, observations=observations)

        # response = call_model(...)
        usage.steps += 1
        # usage.tokens += response_tokens
        # usage.cost_usd += estimated_cost
        # if response is final: return RunResult(StopReason.COMPLETED, response.answer, usage, observations)

        reason = check_budget(budget, usage, started_at=started_at)
        if reason:
            return stopped_result(reason, usage=usage, observations=observations)

        # fingerprint = action_fingerprint(call.name, call.arguments)
        # fingerprints.append(fingerprint)
        # if repeated_too_often(fingerprints, max_same_action=budget.max_same_action):
        #     return stopped_result(StopReason.REPEATED_ACTION, usage=usage, observations=observations)
        # usage.tool_calls += 1
        # observation = execute_tool(...)
        # observations.append(observation)
        # update usage.no_progress based on whether useful state changed

        raise NotImplementedError("Fill this section using the Lesson 3 agent loop")


## Exercises

1. Integrate the budget objects into the Lesson 3 travel agent.
2. Set `max_tool_calls=1` for a goal that needs weather and sunset; verify termination before the second tool.
3. Simulate three identical weather requests and stop with `REPEATED_ACTION`.
4. Use a fake clock to test the time budget without sleeping.
5. Add a mock token/cost accumulator and return a graceful partial result at the ceiling.

Next: **Lesson 8 — Human-in-the-Loop**.
